<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Box Plots**


Estimated time needed: **45** minutes


In this lab, you will focus on the visualization of data. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


In this lab you will perform the following:


-   Visualize the distribution of data.

-   Visualize the relationship between two features.

-   Visualize data composition and comparisons using box plots.


### Setup: Connecting to the Database


#### 1. Download the Database File


In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QR9YeprUYhOoLafzlLspAw/survey-results-public.sqlite

#### 2. Connect to the Database


**Install the needed libraries**


In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Connect to the SQLite database
conn = sqlite3.connect('survey-results-public.sqlite')

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Connect to the SQLite database
conn = sqlite3.connect('survey-results-public.sqlite')


## Demo: Basic SQL Queries


#### Demo 1: Count the Number of Rows in the Table


In [ ]:
QUERY = "SELECT COUNT(*) FROM main"
df = pd.read_sql_query(QUERY, conn)
print(df)


#### Demo 2: List All Tables


In [ ]:
QUERY = """
SELECT name as Table_Name 
FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


#### Demo 3: Group Data by Age


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count 
FROM main 
GROUP BY Age 
ORDER BY Age
"""
df_age = pd.read_sql_query(QUERY, conn)
print(df_age)


## Visualizing Data


### Task 1: Visualizing the Distribution of Data


**1. Box Plot of `CompTotal` (Total Compensation)**


Use a box plot to analyze the distribution and outliers in total compensation.


In [ ]:
# Load CompTotal values and drop missing rows
comp_df = pd.read_sql_query("SELECT CompTotal FROM main WHERE CompTotal IS NOT NULL", conn)
comp_df['CompTotal'] = pd.to_numeric(comp_df['CompTotal'], errors='coerce')
comp_values = comp_df['CompTotal'].dropna()
comp_values = comp_values[comp_values > 0]

# Trim extreme outliers for better visualization in the raw scale
trimmed = comp_values[comp_values <= comp_values.quantile(0.99)]
plt.figure(figsize=(10, 6))
plt.boxplot(trimmed, vert=True, showfliers=False)
plt.title('Distribution of Total Compensation (CompTotal) — trimmed to 99th percentile')
plt.xticks([1], ['CompTotal'])
plt.ylabel('CompTotal')
plt.grid(True)
plt.show()

# Use a log scale to visualize the full skewed distribution
plt.figure(figsize=(10, 6))
plt.boxplot(np.log10(comp_values), vert=True, showfliers=False)
plt.title('Distribution of Total Compensation (CompTotal) — log10 scale')
plt.xticks([1], ['CompTotal'])
plt.ylabel('log10(CompTotal)')
plt.grid(True, which='both', axis='y')
plt.show()

**2. Box Plot of Age (converted to numeric values)**


Convert the `Age` column into numerical values and visualize the distribution.


In [ ]:
# Load age data and map categorical age ranges to numeric midpoints
age_df = pd.read_sql_query("SELECT Age FROM main WHERE Age IS NOT NULL AND Age != ''", conn)

# Map age range categories to numeric midpoints
age_mapping = {
    'Under 18 years old': 15,
    '18-24 years old': 21,
    '25-34 years old': 29.5,
    '35-44 years old': 39.5,
    '45-54 years old': 49.5,
    '55-64 years old': 59.5,
    '65 years or older': 70
}

age_numeric = age_df['Age'].map(age_mapping).dropna()

plt.figure(figsize=(10, 6))
plt.boxplot(age_numeric, vert=True)
plt.title('Distribution of Age (Years) — From Categorical Ranges')
plt.xticks([1], ['Age'])
plt.ylabel('Age (years, mapped from ranges)')
plt.grid(True)
plt.show()

print(f"Age statistics:\nCount: {len(age_numeric)}\nMean: {age_numeric.mean():.1f}\nMedian: {age_numeric.median():.1f}")
print(f"Min: {age_numeric.min():.1f}, Max: {age_numeric.max():.1f}")

### Task 2: Visualizing Relationships in Data


**1. Box Plot of `CompTotal` Grouped by Age Groups:**


Visualize the distribution of compensation across different age groups.


In [ ]:
# Load and prepare data - map age ranges to numeric values
age_mapping = {
    'Under 18 years old': 15,
    '18-24 years old': 21,
    '25-34 years old': 29.5,
    '35-44 years old': 39.5,
    '45-54 years old': 49.5,
    '55-64 years old': 59.5,
    '65 years or older': 70
}

# Load raw data
raw_df = pd.read_sql_query("""
    SELECT Age, CompTotal
    FROM main
    WHERE Age IS NOT NULL AND CompTotal IS NOT NULL
""", conn)

# Map age ranges to numeric and convert CompTotal
raw_df['Age_Numeric'] = raw_df['Age'].map(age_mapping)
raw_df['CompTotal'] = pd.to_numeric(raw_df['CompTotal'], errors='coerce')
raw_df = raw_df.dropna()

# Create age group bins based on mapped numeric values
def age_group_from_numeric(age):
    if age < 23:
        return 'Under 30'
    elif age < 42.5:
        return '30-40'
    elif age < 54.5:
        return '41-50'
    else:
        return 'Over 50'

raw_df['AgeGroup'] = raw_df['Age_Numeric'].apply(age_group_from_numeric)

# Group data by age group
data_by_group = [raw_df[raw_df['AgeGroup'] == group]['CompTotal'].values 
                  for group in ['Under 30', '30-40', '41-50', 'Over 50']]

# Trim extreme outliers (99th percentile) for better visualization
data_trimmed = []
for data in data_by_group:
    if len(data) > 0:
        q99 = np.percentile(data, 99)
        data_trimmed.append(data[data <= q99])
    else:
        data_trimmed.append(data)

plt.figure(figsize=(12, 6))
plt.boxplot(data_trimmed, labels=['Under 30', '30-40', '41-50', 'Over 50'], showfliers=False)
plt.title('Compensation Distribution by Age Group (99th percentile trimmed)')
plt.xlabel('Age Groups')
plt.ylabel('CompTotal')
plt.grid(True, alpha=0.3)
plt.show()

**2. Box Plot of `CompTotal` Grouped by Job Satisfaction (`JobSatPoints_6`):**


Examine how compensation varies based on job satisfaction levels.


In [ ]:
# Load compensation by job satisfaction levels
sat_df = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN JobSatPoints_6 IS NULL OR JobSatPoints_6 = '' THEN 'Unknown'
            WHEN CAST(JobSatPoints_6 AS REAL) < 3 THEN 'Low Satisfaction'
            WHEN CAST(JobSatPoints_6 AS REAL) BETWEEN 3 AND 5 THEN 'Medium Satisfaction'
            ELSE 'High Satisfaction'
        END as SatisfactionLevel,
        CompTotal
    FROM main
    WHERE CompTotal IS NOT NULL AND JobSatPoints_6 IS NOT NULL
""", conn)

sat_df['CompTotal'] = pd.to_numeric(sat_df['CompTotal'], errors='coerce')
sat_df = sat_df.dropna()

# Group data
sat_groups = ['Low Satisfaction', 'Medium Satisfaction', 'High Satisfaction']
data_by_sat = [sat_df[sat_df['SatisfactionLevel'] == sat]['CompTotal'].values for sat in sat_groups]

# Trim outliers (99th percentile)
data_sat_trimmed = []
for data in data_by_sat:
    if len(data) > 0:
        q99 = np.percentile(data, 99)
        data_sat_trimmed.append(data[data <= q99])
    else:
        data_sat_trimmed.append(data)

plt.figure(figsize=(12, 6))
plt.boxplot(data_sat_trimmed, labels=sat_groups, showfliers=False)
plt.title('Compensation Distribution by Job Satisfaction Level (99th percentile trimmed)')
plt.xlabel('Job Satisfaction Level')
plt.ylabel('CompTotal')
plt.xticks(rotation=15)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Task 3: Visualizing the Composition of Data


**1. Box Plot of `ConvertedCompYearly` for the Top 5 Developer Types:**


Analyze compensation across the top 5 developer roles.


In [ ]:
# Get top 5 developer types
top_dev_types = pd.read_sql_query("""
    SELECT DevType, COUNT(*) as count 
    FROM main 
    WHERE DevType IS NOT NULL AND DevType != ''
    GROUP BY DevType 
    ORDER BY count DESC 
    LIMIT 5
""", conn)

dev_types_list = top_dev_types['DevType'].tolist()
placeholders = ','.join([f"'{dt.replace(chr(39), chr(39)*2)}'" for dt in dev_types_list])
dev_type_compensation = pd.read_sql_query(f"""
    SELECT DevType, ConvertedCompYearly 
    FROM main 
    WHERE DevType IN ({placeholders})
    AND ConvertedCompYearly IS NOT NULL
""", conn)

dev_type_compensation['ConvertedCompYearly'] = pd.to_numeric(dev_type_compensation['ConvertedCompYearly'], errors='coerce')
dev_type_compensation = dev_type_compensation.dropna()

# Group data by developer type
data_by_devtype = [dev_type_compensation[dev_type_compensation['DevType'] == dt]['ConvertedCompYearly'].values 
                   for dt in dev_types_list]

# Trim outliers (99th percentile)
data_dev_trimmed = []
for data in data_by_devtype:
    if len(data) > 0:
        q99 = np.percentile(data, 99)
        data_dev_trimmed.append(data[data <= q99])
    else:
        data_dev_trimmed.append(data)

plt.figure(figsize=(14, 6))
plt.boxplot(data_dev_trimmed, labels=dev_types_list, showfliers=False)
plt.title('Annual Compensation Distribution for Top 5 Developer Types (99th percentile trimmed)')
plt.xlabel('Developer Types')
plt.ylabel('ConvertedCompYearly')
plt.xticks(rotation=15, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**2. Box Plot of `CompTotal` for the Top 5 Countries:**


Analyze compensation across respondents from the top 5 countries.


In [ ]:
# Get top 5 countries
top_countries = pd.read_sql_query("""
    SELECT Country, COUNT(*) as count 
    FROM main 
    WHERE Country IS NOT NULL AND Country != ''
    GROUP BY Country 
    ORDER BY count DESC 
    LIMIT 5
""", conn)

countries_list = top_countries['Country'].tolist()
placeholders = ','.join([f"'{c.replace(chr(39), chr(39)*2)}'" for c in countries_list])
country_compensation = pd.read_sql_query(f"""
    SELECT Country, CompTotal 
    FROM main 
    WHERE Country IN ({placeholders})
    AND CompTotal IS NOT NULL
""", conn)

country_compensation['CompTotal'] = pd.to_numeric(country_compensation['CompTotal'], errors='coerce')
country_compensation = country_compensation.dropna()

# Group data by country
data_by_country = [country_compensation[country_compensation['Country'] == c]['CompTotal'].values 
                   for c in countries_list]

# Trim outliers (99th percentile)
data_country_trimmed = []
for data in data_by_country:
    if len(data) > 0:
        q99 = np.percentile(data, 99)
        data_country_trimmed.append(data[data <= q99])
    else:
        data_country_trimmed.append(data)

plt.figure(figsize=(12, 6))
plt.boxplot(data_country_trimmed, labels=countries_list, showfliers=False)
plt.title('Compensation Distribution for Top 5 Countries (99th percentile trimmed)')
plt.xlabel('Countries')
plt.ylabel('CompTotal')
plt.xticks(rotation=15, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Task 4: Visualizing Comparison of Data


**1. Box Plot of CompTotal Across Employment Types:**


Analyze compensation for different employment types.


In [ ]:
# Load employment data with proper filtering
employment_df = pd.read_sql_query("""
    SELECT Employment, CompTotal 
    FROM main 
    WHERE Employment IS NOT NULL AND Employment != '' AND CompTotal IS NOT NULL
""", conn)

employment_df['CompTotal'] = pd.to_numeric(employment_df['CompTotal'], errors='coerce')
employment_df = employment_df.dropna()

# Remove extreme global outliers (values > 99.5th percentile)
q995_global = employment_df['CompTotal'].quantile(0.995)
employment_df_clean = employment_df[employment_df['CompTotal'] <= q995_global]

# Get top 5 employment types by COUNT (most common employment statuses)
top_emp_types = employment_df_clean['Employment'].value_counts().head(5).index.tolist()

# Filter data for top 5 employment types
emp_filtered = employment_df_clean[employment_df_clean['Employment'].isin(top_emp_types)]

# Group data by employment type
data_by_emp = [emp_filtered[emp_filtered['Employment'] == emp]['CompTotal'].values 
               for emp in top_emp_types]

# Trim outliers (99th percentile within each employment type)
data_emp_trimmed = []
for data in data_by_emp:
    if len(data) > 0:
        q99 = np.percentile(data, 99)
        data_emp_trimmed.append(data[data <= q99])
    else:
        data_emp_trimmed.append(data)

plt.figure(figsize=(14, 6))
plt.boxplot(data_emp_trimmed, labels=top_emp_types, showfliers=False)
plt.title('Compensation Distribution for Top 5 Most Common Employment Types (99th percentile trimmed)')
plt.xlabel('Employment Types (most frequent)')
plt.ylabel('CompTotal')
plt.xticks(rotation=20, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Top 5 Most Common Employment Types:")
for emp_type in top_emp_types:
    data = emp_filtered[emp_filtered['Employment'] == emp_type]['CompTotal']
    print(f"  {emp_type}: Count={len(data)}, Median=${data.median():,.0f}, Mean=${data.mean():,.0f}")

**2. Box Plot of `YearsCodePro` by Job Satisfaction (`JobSatPoints_6`):**


Examine the distribution of professional coding years by job satisfaction levels.


In [ ]:
# Load years coding experience data by satisfaction
years_code_df = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN JobSatPoints_6 IS NULL OR JobSatPoints_6 = '' THEN 'Unknown'
            WHEN CAST(JobSatPoints_6 AS REAL) < 3 THEN 'Low Satisfaction'
            WHEN CAST(JobSatPoints_6 AS REAL) BETWEEN 3 AND 5 THEN 'Medium Satisfaction'
            ELSE 'High Satisfaction'
        END as SatisfactionLevel,
        YearsCodePro
    FROM main
    WHERE YearsCodePro IS NOT NULL AND JobSatPoints_6 IS NOT NULL
""", conn)

years_code_df['YearsCodePro'] = pd.to_numeric(years_code_df['YearsCodePro'], errors='coerce')
years_code_df = years_code_df.dropna()

# Group data by satisfaction level
sat_levels = ['Low Satisfaction', 'Medium Satisfaction', 'High Satisfaction']
data_by_sat_years = [years_code_df[years_code_df['SatisfactionLevel'] == sat]['YearsCodePro'].values 
                     for sat in sat_levels]

plt.figure(figsize=(12, 6))
plt.boxplot(data_by_sat_years, labels=sat_levels, showfliers=False)
plt.title('Professional Coding Experience Distribution by Job Satisfaction Level')
plt.xlabel('Job Satisfaction Level')
plt.ylabel('Years of Professional Coding (YearsCodePro)')
plt.xticks(rotation=15, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Professional Coding Years Statistics by Satisfaction:")
for sat, data in zip(sat_levels, data_by_sat_years):
    if len(data) > 0:
        print(f"{sat}: Mean={np.mean(data):.2f}, Median={np.median(data):.2f}, Count={len(data)}")

### Final Step: Close the Database Connection


After completing the lab, close the connection to the SQLite database:


In [ ]:
conn.close()

## Summary


In this lab, you used box plots to visualize various aspects of the dataset, focusing on:

- Visualize distributions of compensation and age.

- Explore relationships between compensation, job satisfaction, and professional coding experience.

- Analyze data composition across developer roles and countries.

- Compare compensation across employment types and satisfaction levels.

Box plots provided clear insights into the spread, outliers, and central tendencies of various features in the dataset.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


<!--## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|               
|2024-10-07|1.2|Madhusudan Moole|Reviewed and updated lab|                                                                                      
|2024-10-06|1.0|Raghul Ramesh|Created lab|-->


Copyright © IBM Corporation. All rights reserved.
